# Image Generation with the Image API

So far we only sent text and files *into* a model and got text back. Models can also generate images for us.

In this notebook we turn the recipe we found in the last notebook, the Bacon, Egg & Spinach Breakfast Sandwiches, into an appetizing image using OpenAI's Image API.

Docs:
- [Image generation guide](https://developers.openai.com/api/docs/guides/image-generation)
- [Images API reference](https://developers.openai.com/api/docs/api-reference/images)

## The quick path

For this lesson, we use one API call: `client.images.generate`.

Prompt in, image out. That is enough to understand the basics quickly.

## Setup

In [1]:
import base64
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Image, display
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

## Defining the model

Image generation uses a dedicated image model, not the text models from the earlier notebooks.

Model Information: 
- [All Models](https://developers.openai.com/api/docs/models/all)
- [GPT Image 2](https://developers.openai.com/api/docs/models/gpt-image-2)

In [ ]:
IMAGE_MODEL = "gpt-image-2"

## A small helper to save and show images

The model returns the image as a base64 string. This helper decodes it, writes it to the `files/` folder, and shows it right in the notebook, so we don't repeat that code in every example.

In [3]:
def save_and_show(image_base64, filename):
    image_path = Path("files") / filename
    image_path.write_bytes(base64.b64decode(image_base64))
    display(Image(filename=str(image_path)))
    return image_path

## Our prompt

A good image prompt describes the subject, the style, and the mood. The more concrete you are, the closer the result matches what you had in mind.

Let's describe an appetizing photo of our breakfast sandwich recipe.

In [ ]:
PROMPT = (
    "A realistic appetizing food photo of a bacon, egg, and spinach breakfast sandwich. "
    "Toasted bread rolls layered with crispy bacon, a fried egg, wilted baby spinach, "
    "melted sliced cheese, and a drizzle of hot sauce. "
    "Bright natural light, shallow depth of field, served on a wooden board, "
    "cozy breakfast mood."
)

## Estimate the cost

Image generation costs depend mostly on the model, image size, and quality.

For a quick lesson estimate, we can use the output prices from the [image generation cost calculator](https://developers.openai.com/api/docs/guides/image-generation#calculating-costs). The prompt text also costs a small number of input tokens, but for a short prompt it is usually much smaller than the image output cost.

Pricing can change, so check the [pricing page](https://developers.openai.com/api/docs/pricing) when exact costs matter.

In [ ]:
SIZE = "1024x1024"
QUALITY = "medium"

IMAGE_OUTPUT_COSTS_USD = {
    ("gpt-image-2", "1024x1024", "low"): 0.006,
    ("gpt-image-2", "1024x1024", "medium"): 0.053,
    ("gpt-image-2", "1024x1024", "high"): 0.211,
    ("gpt-image-2", "1024x1536", "low"): 0.005,
    ("gpt-image-2", "1024x1536", "medium"): 0.041,
    ("gpt-image-2", "1024x1536", "high"): 0.165,
    ("gpt-image-2", "1536x1024", "low"): 0.005,
    ("gpt-image-2", "1536x1024", "medium"): 0.041,
    ("gpt-image-2", "1536x1024", "high"): 0.165,
}

estimated_image_cost = IMAGE_OUTPUT_COSTS_USD[(IMAGE_MODEL, SIZE, QUALITY)]

print(f"Estimated image output cost: ${estimated_image_cost:.3f}")
print("Prompt text costs a little extra, but it is usually tiny for a short prompt.")

## Generating an image

This is the most direct way: one call, prompt in, image out.

- `size` sets the resolution. `"auto"` lets the model choose, or pick `"1024x1024"`, `"1536x1024"` (landscape), or `"1024x1536"` (portrait).
- `quality` trades quality for speed and cost: `"low"`, `"medium"`, or `"high"`.

We use a smaller quality here to keep the call cheap and fast.

In [ ]:
response = client.images.generate(
    model=IMAGE_MODEL,
    prompt=PROMPT,
    size=SIZE,
    quality=QUALITY,
)

save_and_show(response.data[0].b64_json, "breakfast-sandwich.png")

## Things worth remembering

- **Images cost more than text.** You pay mostly for image output tokens. Higher `quality` usually means a more expensive image.
- **Use a quick estimate before running the call.** It helps students understand that every generated image spends API credits.
- **Prompt like you describe a photo.** Name the subject, the style, the colors, and the mood.
- **Use `images.generate` for a fresh picture from a prompt.** That is the only call we need for this quick introduction.
- **The result is base64.** Decode it and save it yourself, like our small helper does.